# Structured text and plain-text records - Python

All 7 Python examples from [docs/text.md](https://platob.github.io/yggdryl/text/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and run on any Python 3 kernel with the package
installed:

```console
pip install yggdryl
```

## Plain-text records

In [ ]:
import pathlib
import tempfile

from yggdryl import IOBase, RecordOptions

with tempfile.TemporaryDirectory() as directory:
    source = pathlib.Path(directory) / "app.log"
    source.write_bytes(b"  [INFO] id=7 first  \r\n[WARN] id=9 second\n")

    options = RecordOptions("text/plain")
    options.header = r"\[(?<level>[A-Z]+)\] id=(?<id>\d+)"
    options.lstrip = r"^\s+"
    options.rstrip = r"\s+$"

    rows = list(IOBase(source).read_records(options=options))
    assert [row["rownum"] for row in rows] == [1, 2]
    assert [row["body"] for row in rows] == [b"first", b"second"]
    assert [row["id"] for row in rows] == [7, 9]

    target = IOBase(pathlib.Path(directory) / "copy.txt")
    target.overwrite_records(
        ({"body": row["body"]} for row in rows),
        options=RecordOptions("text/plain"),
    )
    assert target.read_bytes() == b"first\nsecond\n"

## Raw shared-Scalar access

In [ ]:
from yggdryl import Scalar, json

quote = json.loads('{"symbol":"AAPL","price":12.5}', cls=Scalar)

assert quote["symbol"].as_utf8() == "AAPL"
assert quote.path("price").kind == "f64"
assert quote.set("venue", "XNAS").get("venue").as_utf8() == "XNAS"
assert quote.as_py() == {"price": 12.5, "symbol": "AAPL"}

### Typed `Scalar` families

In [ ]:
from yggdryl import Scalar

assert (Scalar.from_py(40) + 2).as_py() == 42
assert Scalar.decimal(1, 0).divide(Scalar.decimal(2, 0)) == Scalar.decimal(5, 1)

## Field-directed parsing

In [ ]:
from decimal import Decimal

from yggdryl import Field, Scalar, json

amount = Field("amount", "decimal128(8, 2)", nullable=False)
value = json.loads('"12.50"', field=amount, cls=Scalar)

assert value.kind == "d128"
assert value.unscaled == 1_250
assert json.loads('"12.50"', field=amount) == Decimal("12.50")

## Raw document codecs

In [ ]:
from yggdryl import Scalar, codec

value = codec.from_io('{"id":1}', cls=Scalar)

assert isinstance(value, Scalar)
assert value["id"].kind == "u64"
assert codec.into_io(value, format="json", utf8=True) == '{"id":1}'

## Formatting

In [ ]:
from yggdryl import json

pretty = json.dumps({"id": 1}, indent=2)
compact = json.dumps({"id": 1}, indent=None)

assert pretty == b'{\n  "id": 1\n}'
assert compact == b'{"id":1}'

## Placeholders

In [ ]:
from yggdryl import yaml

document = 'host: "{{ HOST }}"\nport: "{{ PORT | default(8080) }}"\n'
value = yaml.loads(document, placeholders={"HOST": "db.internal"})

assert value == {"host": "db.internal", "port": 8080}